# 01. Data and Tables 1–2

All numbers are computed from the Dryad files (`data/`). Run `python scripts/run_pipeline.py` first (processed tables are otherwise built on demand).

In [ ]:
import numpy as np
import pandas as pd
from tbopt import config, data
AGES = config.AGE_GROUPS_40

## Table 1 — national totals (province tables)

In [ ]:
table1 = data.sido_summary_table()
table1

In [ ]:
print(data.table_to_latex(table1))

## District filter

District age-specific deaths are reconstructed as $D_{s,t} = d_{i,t} D_s$ with the province fraction $d_{i,t}$. A district is kept if

1. it has at least one hospital ($H_s > 0$),
2. at least one TB death of any age ($D_s > 0$), and
3. its province reports deaths **and** new cases in every 40+ age group ($D_{s,t} > 0$, $N_{s,t} > 0$ for $t=1..5$).

Condition 3 removes all districts of a province with no deaths in some 40+ age group.

In [ ]:
steps = []
for y in config.YEARS:
    df = data.reconstruct_age(data.load_district(y))
    m1 = df["h"] > 0
    m2 = m1 & (df["D"] > 0)
    m3 = m2 & np.all([df[f"phi_{a}"] > data.PHI_MIN for a in AGES], axis=0)
    prov = df.drop_duplicates("sido").set_index("sido")
    missing = {s: [a for a in AGES if prov.loc[s, f"d{a}"] == 0] for s in prov.index}
    steps.append({"year": y, "all": len(df), "H>0": int(m1.sum()), "+D>0": int(m2.sum()),
                  "+40+ ages": int(m3.sum()),
                  "provinces removed (age groups without deaths)": {s: v for s, v in missing.items() if v}})
pd.DataFrame(steps).set_index("year")

## Table 2 — districts used in the analysis

`D` = all-age deaths, `D_40` = reconstructed 40+ deaths (equals $E$ of the current allocation).

In [ ]:
table2 = data.summary_table()
table2

In [ ]:
print(data.table_to_latex(table2))

## Comparison with the submitted manuscript

The submitted Table 2 was computed from an older version of the processed data; only 2014 and 2022 agree in S, D, H.

In [ ]:
paper1 = pd.DataFrame([(34869, 2136, 330), (32182, 2018, 335), (30892, 2020, 339), (28161, 1678, 344),
                       (26433, 1657, 346), (23821, 1492, 354), (19933, 1222, 360), (18335, 1324, 364),
                       (16264, 1223, 374)], index=config.YEARS, columns=["N", "D", "H"])
paper2 = pd.DataFrame([(143, 32322, 1718, 328), (137, 21683, 1296, 312), (132, 20207, 1255, 306),
                       (143, 19427, 1110, 330), (142, 18145, 1157, 334), (119, 14318, 901, 285),
                       (116, 12079, 736, 276), (113, 10900, 749, 262), (124, 11789, 886, 314)],
                      index=config.YEARS, columns=["S", "N", "D", "H"])
cmp1 = pd.concat({"now": table1, "submitted": paper1}, axis=1).swaplevel(axis=1).sort_index(axis=1)
cmp2 = pd.concat({"now": table2[paper2.columns], "submitted": paper2}, axis=1).swaplevel(axis=1).sort_index(axis=1)
display(cmp1, cmp2)

## Check: processed data equals the Dryad filtered files

In [ ]:
for y in config.YEARS:
    p, f = data.load_processed(y), data.load_district(y, filtered=True)
    assert len(p) == len(f) and all(p[c].sum() == f[c].sum() for c in ["N", "D", "h"]), y
print("OK")

## Check: province age fractions vs. province count tables

The district files carry province fractions `n_t`, `d_t`. Recomputing them from the Dryad province count tables gives the same values except for the cases below (an internal inconsistency of the dataset; the model uses the fractions in the district files).

In [ ]:
rows = []
for y in config.YEARS:
    for kind, used, recomputed in zip("nd", data.sido_age_fractions(y), data.sido_age_fractions_from_counts(y)):
        diff = (used - recomputed.loc[used.index]).abs()
        for s in diff.index[(diff > 1e-6).any(axis=1)]:
            rows.append({"year": y, "fraction": kind, "sido": s, "max_abs_diff": float(diff.loc[s].max())})
pd.DataFrame(rows)